In [10]:
import pandas as pd 
import numpy as np 
import tensorflow as tf


In [11]:
import seaborn as sns

df = sns.load_dataset("diamonds")

In [12]:
df.head()

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


In [13]:
for col in['cut','color','clarity']:
    print(f"\n{col}:")
    print(df[col].unique())


cut:
['Ideal', 'Premium', 'Good', 'Very Good', 'Fair']
Categories (5, str): ['Ideal', 'Premium', 'Very Good', 'Good', 'Fair']

color:
['E', 'I', 'J', 'H', 'F', 'G', 'D']
Categories (7, str): ['D', 'E', 'F', 'G', 'H', 'I', 'J']

clarity:
['SI2', 'SI1', 'VS1', 'VS2', 'VVS2', 'VVS1', 'I1', 'IF']
Categories (8, str): ['IF', 'VVS1', 'VVS2', 'VS1', 'VS2', 'SI1', 'SI2', 'I1']


In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

onehot_encoder=OneHotEncoder()
cat_col=['cut','color','clarity']

In [16]:
preprocess=ColumnTransformer(
    transformers=[
        ('OnehotEncoder', onehot_encoder, cat_col)
    ],
    remainder='passthrough')



In [18]:
x=df.drop('price', axis=1)
y=df['price']

In [19]:
x=preprocess.fit_transform(x)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
x_scaled=scaler.fit_transform(x)

In [25]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y, test_size=0.2, random_state=42)

In [24]:
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard 

In [26]:
model=Sequential(
    [
        Dense(64, activation='relu', input_shape=(x_train.shape[1], )),
        Dense(32, activation='relu'),
        Dense(1)
    ]
)

d:\PROJECTS\DEEP LEARNING PROJECT\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [27]:
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

In [29]:
tensor_board=TensorBoard(log_dir='log_ann', histogram_freq=1)

In [30]:
early_stopping=EarlyStopping(monitor='val_los',patience=10, restore_best_weights=True)

In [31]:
history=model.fit(
    x_train, y_train,
    validation_data=(x_test, y_test),
    epochs=100
)

Epoch 1/100
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - loss: 15993821.0000 - mae: 2927.4204 - val_loss: 11553781.0000 - val_mae: 2560.8508
Epoch 2/100
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 5950541.5000 - mae: 1655.7448 - val_loss: 2796377.2500 - val_mae: 1090.9097
Epoch 3/100
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 2343256.0000 - mae: 1015.7988 - val_loss: 1771246.8750 - val_mae: 910.1801
Epoch 4/100
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 1652849.8750 - mae: 826.5534 - val_loss: 1252742.6250 - val_mae: 711.0953
Epoch 5/100
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 1205825.6250 - mae: 609.7756 - val_loss: 880246.0000 - val_mae: 525.0776
Epoch 6/100
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 1018084.8750 - mae: 524.3016 - val_loss: 776344.7500 - val_mae: 488.6797
Epoch 7/100
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 943096.1875 - mae: 495.5526 - val_loss: 719992.8125 - val_mae: 479.5967
Epoch 8/100
1349/1349 ━━━━━━━━━━━━━━━━

In [32]:
prediction=model.predict(x_test)


338/338 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step


In [33]:
from sklearn.metrics import r2_score
print(r2_score(y_test, prediction))

0.9764475226402283


In [34]:
model.save('reg_model.h5')